In [1]:
import pandas as pd
import joblib
from sklearn.ensemble import IsolationForest
import os

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Paths
processed_path = '/content/drive/MyDrive/FraudDetection_project/processed_data/'
models_path = '/content/drive/MyDrive/FraudDetection_project/saved_models/'
os.makedirs(models_path, exist_ok=True)

print("Loading clustered datasets from Day 5...")
sparkov_train = pd.read_csv(processed_path + 'sparkov_train_clustered.csv')
sparkov_test = pd.read_csv(processed_path + 'sparkov_test_clustered.csv')

# The exact features specified in the Phase 4B diagram
iso_features = ['amt', 'transaction_hour', 'distance_km', 'amount_deviation', 'txn_velocity', 'customer_cluster']

# 1. The Golden Rule: Filter for LEGITIMATE transactions ONLY
print("Isolating legitimate transactions for training...")
legit_train = sparkov_train[sparkov_train['is_fraud'] == 0]

# 2. Fit the Isolation Forest
print("Fitting Isolation Forest model... (This uses parallel processing, but may take a minute)")
# contamination=0.01 tells the model to expect roughly 1% of the data to be anomalous
iso_forest = IsolationForest(contamination=0.01, random_state=42, n_jobs=-1)
iso_forest.fit(legit_train[iso_features])

# 3. Score ALL transactions (Train and Test, Legitimate and Fraud)
print("Calculating anomaly scores for the full datasets...")

# decision_function returns a continuous score (more negative = more anomalous)
sparkov_train['anomaly_score'] = iso_forest.decision_function(sparkov_train[iso_features])
sparkov_test['anomaly_score'] = iso_forest.decision_function(sparkov_test[iso_features])

# predict returns a binary flag (-1 for anomaly, 1 for normal)
sparkov_train['is_anomaly'] = iso_forest.predict(sparkov_train[iso_features])
sparkov_test['is_anomaly'] = iso_forest.predict(sparkov_test[iso_features])

# Let's peek at how well it worked on actual fraud cases
mean_normal_score = sparkov_train[sparkov_train['is_fraud'] == 0]['anomaly_score'].mean()
mean_fraud_score = sparkov_train[sparkov_train['is_fraud'] == 1]['anomaly_score'].mean()

print("\n--- Sanity Check ---")
print(f"Average Anomaly Score for NORMAL transactions: {mean_normal_score:.4f}")
print(f"Average Anomaly Score for FRAUD transactions:  {mean_fraud_score:.4f}")
print("(Fraud scores should be significantly lower/more negative)")

# 4. Save the Model and Enriched Data
print("\nSaving Isolation Forest model and enriched datasets to Drive...")
joblib.dump(iso_forest, models_path + 'isolation_forest.joblib')

# We save these as "enriched" because they now contain every possible engineered feature
sparkov_train.to_csv(processed_path + 'sparkov_train_enriched.csv', index=False)
sparkov_test.to_csv(processed_path + 'sparkov_test_enriched.csv', index=False)

print("Day 6 Deliverables successfully saved!")
print(f"Final Train set shape: {sparkov_train.shape}")

Mounted at /content/drive
Loading clustered datasets from Day 5...
Isolating legitimate transactions for training...
Fitting Isolation Forest model... (This uses parallel processing, but may take a minute)
Calculating anomaly scores for the full datasets...

--- Sanity Check ---
Average Anomaly Score for NORMAL transactions: 0.2019
Average Anomaly Score for FRAUD transactions:  0.0384
(Fraud scores should be significantly lower/more negative)

Saving Isolation Forest model and enriched datasets to Drive...
Day 6 Deliverables successfully saved!
Final Train set shape: (1296675, 33)
